In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import pysr
from pysr import PySRRegressor

import sys
import os
import argparse
import glob

workdir = os.getenv('WORKDIR')
sys.path.append(f'{workdir}/src')

from handlers import trainer, evaluation, annealing, sr_trainer
from handlers.args import setup_argparse
from handlers.model_loader import TorchLoader, SurrogateLoader

from ml_utils import losses
from ml_utils import surrogates
from ml_utils.optimizers import optim

from metrics import complexity, faithfulness, interpretability

from preprocessing.dataloaders import train_load, test_load
from preprocessing.datasets import SimpleIterDataset

from postprocessing.io_writer import _write_outputs_to_root

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import mplhep as mh
import uproot
import awkward as ak
plt.style.use(mh.style.CMS)

from importlib.util import spec_from_file_location, module_from_spec

from weaver.utils.logger import _logger, warn_n_times, _configLogger
import copy
from pprint import pformat
import time
from collections import defaultdict

from main import assemble_loaders

import energyflow

In [21]:
workdir = os.getenv('WORKDIR')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
class Args:
    def __init__(self, **kwargs):
        # defaults
        self.data_train = []
        self.data_test = []
        self.data_val = []
        self.num_workers = 0
        self.num_epochs = 0
        self.data_config = ''
        self.file_fraction = 1
        self.data_fraction = 1
        self.batch_size = 0
        self.local_rank = None
        self.model_prefix = None
        self.lr_finder = None
        self.optimizer_option = []
        self.optimizer = 'ranger'
        self.start_lr = 1e-3
        self.final_lr = 1e-6
        self.lr_scheduler = 'flat+decay'
        self.kl_weight = 0.1
        self.class_weight = 1.0
        self.kl_anneal = False
        self.alpha = 0
        self.beta = 0
        self.gamma = 0
        self.bit_size = None
        self.dr_path = None
        self.dr_network = None
        
        for key, value in kwargs.items():
            setattr(self, key, value)

In [23]:
signals = [
    'TTBar',
    'WToQQ',
    'HToGG'
]

jc_paths = {
    'train': f'{workdir}/datasets/JetClass/Pythia/train_100M',
    'val': f'{workdir}/datasets/JetClass/Pythia/val_5M',
    'test': f'{workdir}/datasets/JetClass/Pythia/test_20M'
}

num_classes = 2
background = '/ZJetsToNuNu_*.root'

datasets = {signal: {} for signal in signals}

for signal in signals:
    for name, path in jc_paths.items():
    
        signal_glob = f'/{signal}_*.root'
        signal_files = glob.glob(path+signal_glob)
        background_files = glob.glob(path+background)
    
        datasets[signal][name] = signal_files + background_files

In [24]:
model_dir = f'{workdir}/outputs/models'
sr_dir = f'{workdir}/outputs/sr_runs'

config_paths = {signal: f'{workdir}/data_config/JetClass/JetClass_{signal}.yaml' for signal in signals}
fig_path = f'{workdir}/figures'

In [25]:
def register_models(signal):
    model_registry = {
        'TTBar': [
            SurrogateLoader(
                model_name='ParT-S',
                teacher_name='ParT',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/TTBar/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['TTBar'],
                sr_path=f'{sr_dir}/TTBar/Surrogate/Surrogate_M7-SRPART_ParT_BVAE_SR',
                equations=[27, 18]
            ),
            SurrogateLoader(
                model_name='ResNet-S',
                teacher_name='ResNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/TTBar/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['TTBar'],
                sr_path=f'{sr_dir}/TTBar/Surrogate/Surrogate_M10-SRRESNET_ResNet_BVAE_SR',
                equations=[19, 15]
            ),
            SurrogateLoader(
                model_name='PN-S',
                teacher_name='ParticleNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/TTBar/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['TTBar'],
                sr_path=f'{sr_dir}/TTBar/Surrogate/Surrogate_M13-SRPARTICLENET_ParticleNet_BVAE_SR',
                equations=[16, 16]
            ),
        ],
        'WToQQ': [
            SurrogateLoader(
                model_name='ParT-S',
                teacher_name='ParT',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/WToQQ/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['WToQQ'],
                sr_path=f'{sr_dir}/WToQQ/Surrogate/Surrogate_M7-SRPART_ParT_BVAE_SR',
                equations=[21, 25]
            ),
            SurrogateLoader(
                model_name='ResNet-S',
                teacher_name='ResNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/WToQQ/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['WToQQ'],
                sr_path=f'{sr_dir}/WToQQ/Surrogate/Surrogate_M10-SRRESNET_ResNet_BVAE_SR',
                equations=[19, 22]
            ),
            SurrogateLoader(
                model_name='PN-S',
                teacher_name='ParticleNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/WToQQ/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['WToQQ'],
                sr_path=f'{sr_dir}/WToQQ/Surrogate/Surrogate_M13-SRPARTICLENET_ParticleNet_BVAE_SR',
                equations=[15, 15]
            ),
        ],
        'HToGG': [
            SurrogateLoader(
                model_name='ParT-S',
                teacher_name='ParT',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/HToGG/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['HToGG'],
                sr_path=f'{sr_dir}/HToGG/Surrogate/Surrogate_M7-SRPART_ParT_BVAE_SR',
                equations=[19, 17]
            ),
            SurrogateLoader(
                model_name='ResNet-S',
                teacher_name='ResNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/HToGG/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['HToGG'],
                sr_path=f'{sr_dir}/HToGG/Surrogate/Surrogate_M10-SRRESNET_ResNet_BVAE_SR',
                equations=[16, 16]
            ),
            SurrogateLoader(
                model_name='PN-S',
                teacher_name='ParticleNet',
                vae_network=f'{workdir}/wrappers/vae.py',
                vae_path=f'{model_dir}/HToGG/BVAE/BVAE_M6-BVAE_DR_epoch-4_state.pt',
                data_config=config_paths['HToGG'],
                sr_path=f'{sr_dir}/HToGG/Surrogate/Surrogate_M13-SRPARTICLENET_ParticleNet_BVAE_SR',
                equations=[23, 16]
            ),
        ]
    }

    return model_registry[signal]

In [28]:
_model_dicts = {signal: {} for signal in signals}

In [29]:
models = register_models(signal)

In [32]:
yaml_config = f'{workdir}/data_config/JetClass/JetClass_{signal}.yaml'
args = Args(
    data_train = datasets[signal]['train'],
    data_val = datasets[signal]['val'],
    data_test = datasets[signal]['test'],
    data_config = yaml_config,
    batch_size = 128,
    file_fraction = 1,
    data_fraction = 0.001,
)

loss_fn = torch.nn.CrossEntropyLoss()

loader_dict = assemble_loaders(args)

In [40]:
loader_dict

{'train': <torch.utils.data.dataloader.DataLoader at 0x7ac43c1cbc70>,
 'val': <torch.utils.data.dataloader.DataLoader at 0x7ac2066d7ac0>,
 'test': {'': functools.partial(<function test_load.<locals>.get_test_loader at 0x7ac206b49750>, '')}}

In [55]:
loader_dict['train'].dataset.config.input_dicts['pf_vectors']

['part_px', 'part_py', 'part_pz', 'part_energy']

In [51]:
_model_dicts = {signal: {} for signal in signals}

skip = []
event='none'
order = 0
segments = 50

for signal in signals:
    if signal in skip:
        continue
    print(f'WORKING ON: {signal}')
    models = register_models(signal)

    yaml_config = f'{workdir}/data_config/JetClass/JetClass_{signal}.yaml'
    args = Args(
        data_train = datasets[signal]['train'],
        data_val = datasets[signal]['val'],
        data_test = datasets[signal]['test'],
        data_config = yaml_config,
        batch_size = 128,
        file_fraction = 1,
        data_fraction = 0.001,
    )

    loss_fn = torch.nn.CrossEntropyLoss()

    loader_dict = assemble_loaders(args)
    
    for model_loader in models:
        model = copy.deepcopy(model_loader.load()).to(device)
        name = model_loader.get_label()
        print(f'Currently working on {name}')
        eq_list = model_loader.fetch_equations()
        tester = evaluation.SurrogateStats(
            loss=loss_fn,
            eq_list = eq_list,
            model=model,
            device=device,
            loader=loader_dict['test'],
            split='test'
        )
        print(f'Initialized Classification Stats for {name}')

        with torch.no_grad():
            test_dict = tester.run()
        if isinstance(model_loader, SurrogateLoader):
            test_dict['complexity']['num_params'][0] -= complexity.total_params(model.dr)[0]
        _model_dicts[signal][name] = test_dict

        del model
        del model_loader
        torch.cuda.empty_cache()

Attempting to load model from /home/alang/ppCodes/part_gp/outputs/sr_runs/TTBar/Surrogate/Surrogate_M7-SRPART_ParT_BVAE_SR/checkpoint.pkl...


WORKING ON: TTBar


NameError: name 'Surrogate' is not defined

In [8]:
pns_inputs = copy.deepcopy(_model_dicts['TTBar']['PN-S']['inputs'])
pns_latents = _model_dicts['TTBar']['PN-S']['preds']['latents']

In [9]:
jet_features = {}
feature_names = []

for key, value in pns_inputs.items():
    if key == 'mask' or key=='features' or key=='p4':
        continue

    jet_features[key] = value
    feature_names.append(key)

In [10]:
dlp4 = pns_inputs['p4']
dlf = pns_inputs['features']

part_px = dlp4[:, 0]
part_py = dlp4[:, 1]
part_pz = dlp4[:, 2]
part_pt = np.hypot(part_px, part_py)
part_e = dlp4[:, 3]
part_deta = dlf[:, 5]
part_dphi = dlf[:, 6]

def wrap_phi(phi):
    return (phi + np.pi) % (2 * np.pi) - np.pi

part_eta = part_deta + jet_features['jet_eta'].reshape(4000, 1)
part_phi = wrap_phi(part_dphi + jet_features['jet_phi'].reshape(4000, 1))

part_m2 = part_e**2 - (part_px**2 + part_py**2 + part_pz**2)
part_m = np.sqrt(np.maximum(part_m2, 0))

In [11]:
hadr_package = [
    part_pt,
    part_eta,
    part_phi,
    part_m
]

ptetaphim = np.concatenate([np.expand_dims(k, axis=1) for k in hadr_package], axis=1)
hadr_coords = np.transpose(ptetaphim, (0, 2, 1))

In [12]:
dmax = 4
beta = 1.0

EFP = energyflow.EFPSet(('d<=', dmax), measure='hadr', beta=beta)

In [29]:
EFP.cols

array(['n', 'e', 'd', 'v', 'k', 'c', 'p', 'h'], dtype='<U1')

In [49]:
EFP.specs[32]

array([6, 4, 4, 2, 0, 2, 2, 4])

In [55]:
EFP.graphs()[32]

[(0, 1), (0, 2), (3, 4), (3, 5)]

In [13]:
efps = EFP.batch_compute(hadr_coords)

In [14]:
efps.shape

(4000, 36)

In [15]:
observables = []
approximable = ['jet_tau1', 'jet_tau2', 'jet_tau3', 'jet_tau4']

for key, value in jet_features.items():
    if key in approximable:
        continue
    observables.append(value)

observables.extend(efps.T)

feature_names = [name for name in feature_names if name not in approximable]

for i in range(len(efps.T)):
    feature_names.append(f'efp_{i}')

In [16]:
x = np.array(observables).transpose((-1, 0))
y = pns_latents[:, :8]

In [104]:
regressor = PySRRegressor(
    maxsize=25,
    niterations=50,
    populations=31,
    population_size = 20,
    ncycles_per_iteration = 100,
    binary_operators=["+", "*", "/", "^"],
    unary_operators = ["log", "sqrt"],
    constraints = {'^': (-1, 1)},
    nested_constraints = {'log': {'log': 0, 'sqrt': 1, '^': 1, '*': 1}, 'sqrt': {'log': 1, 'sqrt': 0}, },
    elementwise_loss="L2DistLoss()")

In [ ]:
regressor.fit(x, y, variable_names=feature_names)

In [6]:
srfeature_paths = {
    'TTBar': f'{workdir}/outputs/sr_runs/TTBar/BVAE_demo_2026-05-25 13:01:55.197530/20260525_130155_upkzr0',
    'WToQQ': f'{workdir}/outputs/sr_runs/WToQQ/BVAE_demo_2026-05-25 16:02:13.827772/20260525_160213_XRgnoX',
    'HToGG': f'{workdir}/outputs/sr_runs/HToGG/BVAE_demo_2026-05-25 19:12:17.434275/20260525_191217_pRZGI1'
}

In [7]:
eq_model = PySRRegressor.from_file(run_directory = srfeature_paths['TTBar'])

Attempting to load model from /home/alang/ppCodes/part_gp/outputs/sr_runs/TTBar/BVAE_demo_2026-05-25 13:01:55.197530/20260525_130155_upkzr0/checkpoint.pkl...


In [26]:
from IPython.display import display, Latex

for signal, path in srfeature_paths.items():
    eq_model = PySRRegressor.from_file(run_directory = path)

    latex_best = eq_model.latex()
    print(f'LaTeX Equations for {signal}:\n')

    for eq in latex_best:
        display(Latex(rf'${eq}$'))

    for eq in latex_best:
        print(f'{eq}\n')
        

Attempting to load model from /home/alang/ppCodes/part_gp/outputs/sr_runs/TTBar/BVAE_demo_2026-05-25 13:01:55.197530/20260525_130155_upkzr0/checkpoint.pkl...


LaTeX Equations for TTBar:



<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

...

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

<IPython.core.display.Latex object>

\log{\left(\frac{jet_{pt}}{efp_{3}^{0.132} jet_{energy}} \right)}

\frac{0.00791 + jet_{energy} \left(-3.11 \cdot 10^{-6}\right)}{efp_{35} + \frac{1.92}{jet_{pt}}} - 1.18

efp_{9}^{efp_{5}} + 1.10 - \frac{1.64 \cdot 10^{3}}{jet_{energy}}

efp_{1} - 1.86 + \frac{1.59}{\frac{jet_{energy}}{-276.} + 1.51} - \frac{6.89}{\frac{jet_{energy}}{-216.} + 1.39}

0.698 - \frac{0.0716}{jet_{nparticles}^{-6.06} \cdot 7.39 \cdot 10^{8} + 0.0418}

- 8.86 \sqrt{efp_{11}} + \frac{efp_{15}}{efp_{18}} + 0.390

0.587 + \frac{1.08}{-0.762 - \frac{1.88}{\left(jet_{nparticles} 0.0277\right)^{14.2}}}

- 1.71 \left(jet_{pt} \left(jet_{energy} 4.42 \cdot 10^{-7} - 0.00119\right) + \left(\left({0.906}^{jet_{pt}}\right)^{jet_{tau1}}\right)^{jet_{tau1}}\right)

